# LoRA fine-tuning: does a small model beat the length baseline at judging a coding agent's closing message?

Part of [lora-appraisal-judge](https://github.com/carlosherreraUMA/lora-appraisal-judge), built to close a stack gap for research-engineer
applications (LoRA/QLoRA, the Hugging Face ecosystem, evaluation-driven model
improvement) and to extend
[situated-appraisal-in-coding-agent-trajectories](https://github.com/carlosherreraUMA/situated-appraisal-in-coding-agent-trajectories)'s
finding that a coding agent's own claim of success is uninformative
(~98% of both failed and successful runs claim success).

**Question:** given only the agent's closing message, can a QLoRA-fine-tuned small
model (`Qwen/Qwen2.5-1.5B-Instruct`, 4-bit) predict whether the task was actually
resolved any better than (a) the majority class, or (b) a trajectory-length
baseline — and does any of that survive a **within-task paired comparison**, which
controls for task difficulty the way the source project's own D11 does?

**To run:** open this notebook on Kaggle (upload it or import from GitHub), turn on
Settings → Accelerator → **GPU T4 x2** (one is enough) and Settings → **Internet → On**
(needed to clone the repo and download the base model), then run all cells top to
bottom.


In [ ]:
# Restrict to a single GPU, even if the "T4 x2" accelerator is selected. Must run
# before torch/CUDA initialize anywhere else in the session (including in a previous
# run of this notebook) — if you have already run a later cell once, restart the
# session (Run -> Restart Session) and run from this cell again for it to take effect.
# See training.py's load_quantized_model_and_tokenizer for why: two visible GPUs plus
# bitsandbytes 4-bit quantization plus Trainer's automatic DataParallel wrapping
# corrupts the quantized weights and crashes with a CUDA "illegal memory access".
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
# Only needed on Kaggle: torch/transformers/datasets ship preinstalled, these don't.
# No bitsandbytes: see training.py's load_model_and_tokenizer for why (its 4-bit
# kernel failed on this T4 and silently fell back to a ~20-50x slower path).
!pip install -q peft accelerate


In [ ]:
# Only needed on Kaggle: torch/transformers/datasets ship preinstalled, these don't.
!pip install -q peft bitsandbytes accelerate


## 1. Load the prepared dataset

Committed to the repo under `data/prepared/` (small — see its own `manifest.json` for provenance and attribution).

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path(REPO_DIR) / "data" / "prepared"

def load_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh]

train = load_jsonl(DATA_DIR / "train.jsonl")
val = load_jsonl(DATA_DIR / "val.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")
test_pairs_ids = json.load(open(DATA_DIR / "test_pairs.json"))

print(f"train={len(train)} val={len(val)} test={len(test)} pairs={len(test_pairs_ids)}")
print(json.load(open(DATA_DIR / "manifest.json")))


## 2. Baselines (no GPU needed — this cell is the honest floor the LoRA result has to beat)

Same code as `tests/test_baselines.py` / `tests/test_metrics.py`, run here on the
real split so the comparison in the results table is apples to apples. Local run on
this dataset (9,000 raw trajectories scanned, 8,128 eligible) already found: the
majority baseline is at chance by construction; the length baseline reaches
AUC 0.638 overall but only 0.491 under the within-task paired comparison — the
naive length signal is confounded by task difficulty, exactly the kind of
confound `situated-appraisal`'s own D24 controls for. See `EXPERIMENTS.md`.

In [ ]:
## 3. Load the base model in bf16 and attach a LoRA adapter

Not 4-bit this time — see `training.py`'s `load_model_and_tokenizer` docstring for
why (the 4-bit kernel failed on this T4 and silently fell back to a much slower
path, most likely the reason the first run took over 8 hours instead of well under
one).

from lora_appraisal_judge.training import (
    LoraTrainingConfig,
    build_lora_model,
    build_tokenized_dataset,
    load_model_and_tokenizer,
)

config = LoraTrainingConfig()  # see training.py for defaults (Qwen2.5-1.5B-Instruct, r=16, ...)
model, tokenizer = load_model_and_tokenizer(config)
model = build_lora_model(model, config)
model.print_trainable_parameters()

train_ds = build_tokenized_dataset(tokenizer, train, config.max_seq_length)
val_ds = build_tokenized_dataset(tokenizer, val, config.max_seq_length)


In [ ]:
## 4. Fine-tune

Loss is masked to the response tokens only, by hand (`CausalLMLabelCollator`,
`labels = -100` on every prompt token) — see `training.py` for why this uses plain
`transformers.Trainer` rather than `trl`'s `SFTTrainer`.

**Checkpointed every 50 steps this time** (`save_total_limit=3`, so only the last
three are kept), and the cell resumes automatically from the latest one if you have
to re-run it — a kernel restart or hitting the session time limit now loses at most
50 steps, not the whole run. **Watch the printed rate for the first minute or two**:
this should run at something like 1+ it/s on a T4 for a 1.5B model at this batch
size. If it is anywhere near as slow as the first attempt (0.02 it/s), it is now
safe to interrupt (Run → Stop, not Restart) and tell Claude, rather than letting it
run unattended for hours again.

import os

from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

from lora_appraisal_judge.training import CausalLMLabelCollator

collator = CausalLMLabelCollator(pad_token_id=tokenizer.pad_token_id)

OUTPUT_DIR = "lora-appraisal-judge-out"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    report_to=[],
    seed=config.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
)

# Resumes automatically if this cell is re-run after an interruption (a checkpoint
# under OUTPUT_DIR from `save_steps` above) — `None` on a first run, so this is safe
# either way.
last_checkpoint = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if last_checkpoint:
    print(f"Resuming from {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)


In [ ]:
from transformers import Trainer, TrainingArguments

from lora_appraisal_judge.training import CausalLMLabelCollator

collator = CausalLMLabelCollator(pad_token_id=tokenizer.pad_token_id)

training_args = TrainingArguments(
    output_dir="lora-appraisal-judge-out",
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    logging_steps=20,
    save_strategy="no",
    bf16=True,
    report_to=[],
    seed=config.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
)
trainer.train()


## 5. Evaluate on the held-out test set

Two readings per example, for two different purposes:
- `resolved_score` (log P(RESOLVED) - log P(UNRESOLVED), teacher-forced): a
  continuous score for `auc` and `paired_separation` — does not depend on the model
  emitting exactly the trained wording.
- `generate_prediction` -> `parse_prediction`: what a real deployment would read,
  for `accuracy` / `balanced_accuracy`.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

from lora_appraisal_judge.prompts import parse_prediction
from lora_appraisal_judge.training import generate_prediction, resolved_score

model.eval()
scores, preds = [], []
for ex in tqdm(test):
    scores.append(resolved_score(model, tokenizer, ex["prompt"]))
    preds.append(parse_prediction(generate_prediction(model, tokenizer, ex["prompt"])))

scores = np.array(scores)
score_by_id = dict(zip((ex["trajectory_id"] for ex in test), scores.tolist()))

results["lora"] = {
    "accuracy": accuracy(y_true, preds),
    "balanced_accuracy": balanced_accuracy(y_true, preds),
    "auc": auc(y_true, scores.tolist()),
    "paired_separation": paired_separation(pairs, score_by_id),
    "n_unparsed_predictions": sum(1 for p in preds if p is None),
}
results


## 6. Save the adapter and the results

Only the LoRA adapter is saved (a few tens of MB), not the base model. Copy the
printed `results` dict into `EXPERIMENTS.md` in the repo afterwards, honestly,
including if `lora.paired_separation` does **not** beat `length.paired_separation`
— that is a real result, in the spirit of the project this extends, not a failed
run to hide.

In [ ]:
model.save_pretrained("lora-appraisal-judge-adapter")
tokenizer.save_pretrained("lora-appraisal-judge-adapter")

with open("results.json", "w") as fh:
    json.dump(results, fh, indent=2)

results
